# Phase A: Dataset Inspection

This notebook inspects the source CSV only. It does not alter data or train a model.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path.cwd().parent / 'EV_Battery_Charging_TR_Dataset_with_Notes (1).csv'
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / 'EV_Battery_Charging_TR_Dataset_with_Notes (1).csv'
if not DATA_PATH.exists():
    raise FileNotFoundError('Place the dataset at the project root before running this notebook.')

df = pd.read_csv(DATA_PATH)
print(f'File: {DATA_PATH.name}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

In [ ]:
inspection = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique(dropna=True),
})
print('Duplicate rows:', df.duplicated().sum())
inspection

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns
df[numeric_columns].describe().T

# IQR counts are flags for review, not automatic removal decisions.
q1 = df[numeric_columns].quantile(0.25)
q3 = df[numeric_columns].quantile(0.75)
iqr = q3 - q1
outlier_counts = ((df[numeric_columns] < q1 - 1.5 * iqr) | (df[numeric_columns] > q3 + 1.5 * iqr)).sum()
outlier_counts.rename('iqr_outlier_count').to_frame()

In [ ]:
target = 'MaxTemp_C'
assert target in df.columns, f'Missing target column: {target}'

target_summary = df[target].describe()
correlations = df[numeric_columns].corr(numeric_only=True)[target].sort_values(ascending=False)
print(target_summary)
correlations.to_frame('pearson_correlation_with_MaxTemp_C')

## Leakage guardrail

Do not train with `MinTemp_C`, `AvgTemp_C`, `TR_Probability`, `BMS_Status`, `EventFlag`, or `Notes` in the initial real-time model. They are directly thermal, risk-derived, or post-event fields. See `reports/dataset_analysis.md` for the full rationale and conservative feature list.